 %md
 # 03 - Profiling e Qualidade da Camada Bronze

 ## MVP-3 — Engenharia de Dados

 Este notebook analisa a qualidade dos dados armazenados
 na camada Bronze.

 ### Objetivos
 - conhecer o volume e a estrutura dos dados;
 - identificar valores nulos;
 - identificar registros duplicados;
 - analisar valores distintos e domínios relevantes;
 - avaliar campos utilizados na integração dos datasets;
 - identificar problemas que deverão ser tratados na camada Prata.

 Nenhum dado é alterado neste notebook.

In [0]:
from pyspark.sql import functions as F


CATALOGO = "mvp3_eng_dados"
SCHEMA_BRONZE = "bronze"


TABELA_IMDB = (
    f"{CATALOGO}.{SCHEMA_BRONZE}.imdb_avaliacoes"
)

TABELA_RT_FILMES = (
    f"{CATALOGO}.{SCHEMA_BRONZE}.rt_filmes"
)

TABELA_RT_AVALIACOES = (
    f"{CATALOGO}.{SCHEMA_BRONZE}.rt_avaliacoes"
)


print("✓ Configuração carregada")

✓ Configuração carregada


In [0]:
df_imdb = spark.table(
    TABELA_IMDB
)

df_rt_filmes = spark.table(
    TABELA_RT_FILMES
)

df_rt_avaliacoes = spark.table(
    TABELA_RT_AVALIACOES
)


print("✓ Tabelas Bronze carregadas")

✓ Tabelas Bronze carregadas


In [0]:
resumo_geral = [
    (
        "IMDb - Avaliações",
        df_imdb.count(),
        len(df_imdb.columns)
    ),
    (
        "RT - Filmes",
        df_rt_filmes.count(),
        len(df_rt_filmes.columns)
    ),
    (
        "RT - Avaliações",
        df_rt_avaliacoes.count(),
        len(df_rt_avaliacoes.columns)
    )
]


df_resumo_geral = spark.createDataFrame(
    resumo_geral,
    [
        "dataset",
        "quantidade_registros",
        "quantidade_colunas"
    ]
)


display(df_resumo_geral)

dataset,quantidade_registros,quantidade_colunas
IMDb - Avaliações,50000,5
RT - Filmes,17712,25
RT - Avaliações,1130017,11


In [0]:
def analisar_nulos(df):

    total = df.count()

    expressoes = []

    for coluna in df.columns:

        expressoes.append(
            F.sum(
                F.when(
                    F.col(coluna).isNull(),
                    1
                ).otherwise(0)
            ).alias(coluna)
        )

    resultado = (
        df
        .select(expressoes)
        .first()
        .asDict()
    )

    dados = []

    for coluna, quantidade in resultado.items():

        percentual = (
            quantidade / total * 100
            if total > 0
            else 0
        )

        dados.append(
            (
                coluna,
                quantidade,
                round(percentual, 2)
            )
        )

    return spark.createDataFrame(
        dados,
        [
            "coluna",
            "quantidade_nulos",
            "percentual_nulos"
        ]
    )

In [0]:
print("IMDb - Valores nulos")

df_nulos_imdb = analisar_nulos(
    df_imdb
)

display(
    df_nulos_imdb
    .orderBy(
        F.desc("percentual_nulos")
    )
)

IMDb - Valores nulos


coluna,quantidade_nulos,percentual_nulos
review,0,0.0
sentiment,0,0.0
_fonte,0,0.0
_arquivo_origem,0,0.0
_data_ingestao,0,0.0


In [0]:
print("Rotten Tomatoes - Filmes - Valores nulos")

df_nulos_rt_filmes = analisar_nulos(
    df_rt_filmes
)

display(
    df_nulos_rt_filmes
    .orderBy(
        F.desc("percentual_nulos")
    )
)

Rotten Tomatoes - Filmes - Valores nulos


coluna,quantidade_nulos,percentual_nulos
critics_consensus,8578,48.43
authors,1542,8.71
original_release_date,1166,6.58
production_company,499,2.82
audience_status,448,2.53
streaming_release_date,384,2.17
actors,352,1.99
movie_info,321,1.81
runtime,314,1.77
audience_count,297,1.68


In [0]:
print("Rotten Tomatoes - Avaliações - Valores nulos")

df_nulos_rt_avaliacoes = analisar_nulos(
    df_rt_avaliacoes
)

display(
    df_nulos_rt_avaliacoes
    .orderBy(
        F.desc("percentual_nulos")
    )
)

Rotten Tomatoes - Avaliações - Valores nulos


coluna,quantidade_nulos,percentual_nulos
review_score,305936,27.07
review_content,65806,5.82
critic_name,18529,1.64
rotten_tomatoes_link,0,0.0
top_critic,0,0.0
publisher_name,0,0.0
review_type,0,0.0
review_date,0,0.0
_fonte,0,0.0
_arquivo_origem,0,0.0


In [0]:
def contar_duplicados(df):

    total = df.count()

    distintos = (
        df
        .drop(
            "_fonte",
            "_arquivo_origem",
            "_data_ingestao"
        )
        .distinct()
        .count()
    )

    return total - distintos


duplicados_imdb = contar_duplicados(
    df_imdb
)

duplicados_rt_filmes = contar_duplicados(
    df_rt_filmes
)

duplicados_rt_avaliacoes = contar_duplicados(
    df_rt_avaliacoes
)


print(
    f"IMDb - duplicados: "
    f"{duplicados_imdb:,}"
)

print(
    f"RT Filmes - duplicados: "
    f"{duplicados_rt_filmes:,}"
)

print(
    f"RT Avaliações - duplicados: "
    f"{duplicados_rt_avaliacoes:,}"
)

IMDb - duplicados: 418
RT Filmes - duplicados: 0
RT Avaliações - duplicados: 119,471


In [0]:
display(
    df_imdb
    .groupBy("sentiment")
    .count()
    .orderBy(
        F.desc("count")
    )
)

sentiment,count
positive,25000
negative,25000


In [0]:
display(
    df_rt_avaliacoes
    .groupBy("review_type")
    .count()
    .orderBy(
        F.desc("count")
    )
)

review_type,count
Fresh,720210
Rotten,409807


In [0]:
display(
    df_rt_avaliacoes
    .groupBy("top_critic")
    .count()
    .orderBy(
        F.desc("count")
    )
)

top_critic,count
False,841481
True,288536


In [0]:
total_rt_filmes = df_rt_filmes.count()

nulos_chave_filmes = (
    df_rt_filmes
    .filter(
        F.col(
            "rotten_tomatoes_link"
        ).isNull()
    )
    .count()
)


print(
    f"Total de filmes: "
    f"{total_rt_filmes:,}"
)

print(
    f"Chaves nulas: "
    f"{nulos_chave_filmes:,}"
)

Total de filmes: 17,712
Chaves nulas: 0


In [0]:
chaves_distintas = (
    df_rt_filmes
    .select(
        "rotten_tomatoes_link"
    )
    .distinct()
    .count()
)


print(
    f"Registros de filmes: "
    f"{total_rt_filmes:,}"
)

print(
    f"Chaves distintas: "
    f"{chaves_distintas:,}"
)


if total_rt_filmes == chaves_distintas:
    print("✓ Chave única em RT Filmes")
else:
    print("⚠ Existem chaves repetidas em RT Filmes")

Registros de filmes: 17,712
Chaves distintas: 17,712
✓ Chave única em RT Filmes


In [0]:
df_avaliacoes_sem_filme = (
    df_rt_avaliacoes.alias("a")
    .join(
        df_rt_filmes.alias("f"),
        F.col(
            "a.rotten_tomatoes_link"
        )
        ==
        F.col(
            "f.rotten_tomatoes_link"
        ),
        "left_anti"
    )
)


qtd_avaliacoes_sem_filme = (
    df_avaliacoes_sem_filme.count()
)


print(
    "Avaliações sem filme correspondente: "
    f"{qtd_avaliacoes_sem_filme:,}"
)

Avaliações sem filme correspondente: 130


In [0]:
display(
    df_rt_avaliacoes
    .groupBy("review_score")
    .count()
    .orderBy(
        F.desc("count")
    )
    .limit(30)
)

review_score,count
null,305936
3/5,90273
4/5,83659
3/4,72366
2/5,60174
2/4,47546
2.5/4,44994
3.5/4,34371
3.5/5,30885
B,24358


In [0]:
display(
    df_rt_avaliacoes
    .select(
        "review_date"
    )
    .where(
        F.col("review_date").isNotNull()
    )
    .distinct()
    .orderBy(
        "review_date"
    )
    .limit(20)
)

review_date
1800-01-01
1923-01-01
1927-08-12
1927-09-23
1928-04-21
1929-02-01
1931-01-26
1932-01-01
1933-01-01
1934-02-22


In [0]:
def contar_textos_vazios(
    df,
    coluna
):

    return (
        df
        .filter(
            F.trim(
                F.col(coluna)
            ) == ""
        )
        .count()
    )


vazios_imdb = contar_textos_vazios(
    df_imdb,
    "review"
)

vazios_rt = contar_textos_vazios(
    df_rt_avaliacoes,
    "review_content"
)


print(
    f"IMDb - reviews vazias: "
    f"{vazios_imdb:,}"
)

print(
    f"RT - reviews vazias: "
    f"{vazios_rt:,}"
)

IMDb - reviews vazias: 0
RT - reviews vazias: 0


In [0]:
indicadores = [
    (
        "IMDb",
        df_imdb.count(),
        duplicados_imdb,
        vazios_imdb
    ),
    (
        "RT Filmes",
        df_rt_filmes.count(),
        duplicados_rt_filmes,
        nulos_chave_filmes
    ),
    (
        "RT Avaliações",
        df_rt_avaliacoes.count(),
        duplicados_rt_avaliacoes,
        vazios_rt
    )
]


df_indicadores = spark.createDataFrame(
    indicadores,
    [
        "dataset",
        "registros",
        "duplicados",
        "problemas_campo_principal"
    ]
)


display(df_indicadores)

dataset,registros,duplicados,problemas_campo_principal
IMDb,50000,418,0
RT Filmes,17712,0,0
RT Avaliações,1130017,119471,0


In [0]:
print("=" * 60)
print("PROFILING DA CAMADA BRONZE CONCLUÍDO")
print("=" * 60)


print("\nDatasets analisados:")
print("✓ IMDb - Avaliações")
print("✓ Rotten Tomatoes - Filmes")
print("✓ Rotten Tomatoes - Avaliações")


print("\nDimensões avaliadas:")
print("✓ Volume")
print("✓ Estrutura")
print("✓ Completude")
print("✓ Duplicidades")
print("✓ Domínios")
print("✓ Chaves")
print("✓ Integridade referencial")
print("✓ Campos textuais")
print("✓ Formatos de score e data")


print("\nAlterações nos dados:")
print("NENHUMA")


print("\nStatus:")
print("✓ PROFILING BRONZE CONCLUÍDO")


print("\nPróximas etapas:")
print("04_imdb_prata")
print("05_rt_filmes_prata")
print("06_rt_avaliacoes_prata")

PROFILING DA CAMADA BRONZE CONCLUÍDO

Datasets analisados:
✓ IMDb - Avaliações
✓ Rotten Tomatoes - Filmes
✓ Rotten Tomatoes - Avaliações

Dimensões avaliadas:
✓ Volume
✓ Estrutura
✓ Completude
✓ Duplicidades
✓ Domínios
✓ Chaves
✓ Integridade referencial
✓ Campos textuais
✓ Formatos de score e data

Alterações nos dados:
NENHUMA

Status:
✓ PROFILING BRONZE CONCLUÍDO

Próximas etapas:
04_imdb_prata
05_rt_filmes_prata
06_rt_avaliacoes_prata
